# Step 5 — Consolidate the Datasets

This notebook combines the outputs of Steps 1–4 into **three datasets** that the application uses:

| Dataset | One row per | Built from |
|---|---|---|
| `schools` | school (all statuses) | Step 1 school data + Step 2 square footage + Step 3 climate zone, joined on `school_id` |
| `districts` | district | the `schools` dataset: district ID, name and state (used by the application's district search) |
| `bpd_eui_benchmarks` | climate zone × BPD building type | Step 4 |

**Outputs** (in `data/processed/`):
- `schools_2024_25.csv`, `districts_2024_25.csv`, `bpd_eui_benchmarks.csv`
- `school_energy_datasets_2024_25.xlsx` — the same three datasets as three sheets of one workbook

## Inputs

| Step | File in `data/interim/` |
|---|---|
| 1 | `schools_2024_25.csv` |
| 2 | `school_sqft_2024_25_all.csv` if the national run was done, otherwise one `school_sqft_2024_25_<STATE>.csv` |
| 3 | `school_climate_zone_2024_25.csv` |
| 4 | `bpd_eui_benchmarks.csv` |

Square footage and climate zone exist only for **operational** schools. Schools outside the square-footage run's
scope (e.g. every school outside New York after a New York-only run) get a **blank** square footage, never 0.

## 1. Setup

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)


def check(condition: bool, failure_message: str) -> None:
    # Stops the notebook with a plain-language message instead of continuing with bad data.
    if not condition:
        raise ValueError(failure_message)


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

SCHOOLS_FILE = INTERIM_DIR / "schools_2024_25.csv"
CLIMATE_ZONE_FILE = INTERIM_DIR / "school_climate_zone_2024_25.csv"
BENCHMARKS_FILE = INTERIM_DIR / "bpd_eui_benchmarks.csv"

# BPD's name for the K-12 school category in the Step 4 downloads. Every school is benchmarked against this category
# (section 6). Check the building types printed in section 2 and update this if BPD names it differently.
K12_BUILDING_TYPE = "K-12 School"

# Use the national square-footage run if it exists; otherwise the single-state run.
national_sqft_file = INTERIM_DIR / "school_sqft_2024_25_all.csv"
state_sqft_files = sorted(
    path for path in INTERIM_DIR.glob("school_sqft_2024_25_*.csv") if path != national_sqft_file
)
state_file_names = [path.name for path in state_sqft_files]
check(
    national_sqft_file.exists() or len(state_sqft_files) == 1,
    f"Expected school_sqft_2024_25_all.csv or exactly one state file in data/interim; found {state_file_names}.",
)
SQFT_FILE = national_sqft_file if national_sqft_file.exists() else state_sqft_files[0]

for path in (SCHOOLS_FILE, SQFT_FILE, CLIMATE_ZONE_FILE, BENCHMARKS_FILE):
    check(
        path.exists(),
        f"Input not found: {path.relative_to(PROJECT_ROOT)}. Run the step that produces it first.",
    )
print(f"Square-footage input: {SQFT_FILE.name}")

## 2. Load the four step outputs

In [ ]:
TEXT = {"dtype": str, "keep_default_na": False, "na_values": [""]}
step1_schools = pd.read_csv(SCHOOLS_FILE, **TEXT)
step2_sqft = pd.read_csv(SQFT_FILE, **TEXT)
step3_climate = pd.read_csv(CLIMATE_ZONE_FILE, **TEXT)
benchmarks = pd.read_csv(BENCHMARKS_FILE, dtype={"climate_zone": str, "building_type": str})

for name, table in {
    "Step 1 schools": step1_schools,
    "Step 2 square footage": step2_sqft,
    "Step 3 climate zone": step3_climate,
}.items():
    check(table["school_id"].is_unique, f"{name}: a school appears more than once.")
    check(
        table["school_id"].isin(step1_schools["school_id"]).all(),
        f"{name}: contains schools that are not in Step 1.",
    )
check(
    not benchmarks.duplicated(["climate_zone", "building_type"]).any(),
    "Step 4 benchmarks: a climate zone × building type appears twice.",
)

pd.DataFrame(
    {"rows": [len(step1_schools), len(step2_sqft), len(step3_climate), len(benchmarks)]},
    index=[
        "Step 1 schools",
        f"Step 2 square footage ({SQFT_FILE.stem.rsplit('_', 1)[-1]})",
        "Step 3 climate zone",
        "Step 4 benchmarks",
    ],
)

In [ ]:
print("BPD building types in the benchmarks:", sorted(benchmarks["building_type"].unique()))

## 3. `schools`

Step 1 is the master list; every school stays. Square footage (Step 2) and climate zone (Step 3) are attached by
`school_id`. Each join is checked so it cannot add or drop a school.

In [ ]:
SQFT_COLUMNS = [
    "footprint_file_available",
    "building_found",
    "footprint_sqft",
    "building_capture_dates",
    "nearest_building_distance_m",
    "nearest_building_sqft",
    "schools_at_same_point",
    "estimated_sqft",
    "sqft_method",
]
CLIMATE_COLUMNS = ["climate_zone", "climate_zone_method"]

schools = step1_schools.copy()
for name, table in {
    "square footage": step2_sqft[["school_id"] + SQFT_COLUMNS],
    "climate zone": step3_climate[["school_id"] + CLIMATE_COLUMNS],
}.items():
    rows_before = len(schools)
    schools = schools.merge(table, on="school_id", how="left", validate="one_to_one")
    check(
        len(schools) == rows_before,
        f"Joining {name} changed the row count from {rows_before:,} to {len(schools):,}.",
    )

# Label why a school has no square footage, so a blank is never mistaken for a missing calculation.
operational = schools["is_operational"] == "True"
in_sqft_run = schools["school_id"].isin(step2_sqft["school_id"])
schools.loc[~operational, "sqft_method"] = "not_operational"
schools.loc[operational & ~in_sqft_run, "sqft_method"] = "outside_square_footage_run"
schools.loc[~operational, "climate_zone_method"] = "not_operational"

summary = pd.DataFrame(
    {
        "schools": len(schools),
        "operational": operational.sum(),
        "with estimated_sqft": schools["estimated_sqft"].notna().sum(),
        "with climate_zone": schools["climate_zone"].notna().sum(),
        "with both": (schools["estimated_sqft"].notna() & schools["climate_zone"].notna()).sum(),
    },
    index=["schools dataset"],
)
summary

In [ ]:
schools["sqft_method"].value_counts().to_frame("schools")

## 4. `districts`

One row per district that has at least one school, taken from the school data, with school counts.

In [ ]:
schools_with_flag = schools.assign(operational=schools["is_operational"] == "True")
districts = (
    schools_with_flag.groupby(["district_id", "district_name", "admin_state", "admin_state_name"])
    .agg(schools=("school_id", "size"), operational_schools=("operational", "sum"))
    .reset_index()
    .sort_values(["admin_state", "district_name"])
    .reset_index(drop=True)
)
check(districts["district_id"].is_unique, "A district has more than one name or state in the school data.")
check(
    districts["schools"].sum() == len(schools),
    "District school counts do not add up to the number of schools.",
)
print(f"Districts: {len(districts):,}")
districts.head()

## 5. `bpd_eui_benchmarks`

Step 4's table, unchanged: one row per climate zone × BPD building type, with site, electric and fuel EUI statistics.

In [ ]:
benchmarks.head()

## 6. Call-out: how a school is linked to its benchmark

The datasets are **not pre-joined** to benchmarks. The link is made by the application, because which BPD category a
school uses depends on how BPD names its categories at the time of download.

**Rule:** BPD now groups elementary, middle and high schools into one **K-12** category, so **every school is
benchmarked as K-12**:

1. Take the school's `climate_zone` from `schools` (e.g. `5A`).
2. In `bpd_eui_benchmarks`, find the row with that `climate_zone` and `building_type` = the K-12 category
   (`K12_BUILDING_TYPE` in section 1).
3. Read the percentile needed — e.g. `site_eui_p50` for a median estimate, or `electric_eui_p50` and `fuel_eui_p50`
   for the electricity / fuel split.
4. Estimated annual energy (kBtu) = `estimated_sqft` × the chosen EUI.

A school gets no benchmark when it has no `climate_zone`, or when its climate zone has no K-12 row in BPD.
Prekindergarten-only schools are also benchmarked as K-12 under this rule.

The cells below show the lookup on real rows and count how many schools it covers.

In [ ]:
def k12_benchmark(climate_zone: str, statistic: str = "site_eui_p50") -> float | None:
    # The lookup the application performs: K-12 row of the school's climate zone.
    row = benchmarks[
        (benchmarks["climate_zone"] == climate_zone) & (benchmarks["building_type"] == K12_BUILDING_TYPE)
    ]
    return None if row.empty else float(row.iloc[0][statistic])


k12_rows = benchmarks[benchmarks["building_type"] == K12_BUILDING_TYPE]
if k12_rows.empty:
    print(
        f"No '{K12_BUILDING_TYPE}' rows in the benchmarks. Building types present: {sorted(benchmarks['building_type'].unique())}."
    )
    print("Set K12_BUILDING_TYPE in section 1 to BPD's K-12 category name and re-run.")
else:
    has_benchmark = operational & schools["climate_zone"].isin(k12_rows["climate_zone"])
    print(
        f"Operational schools whose climate zone has a K-12 benchmark: {has_benchmark.sum():,} of {operational.sum():,}"
    )
    example = schools[has_benchmark & schools["estimated_sqft"].notna()].head(3).copy()
    example["site_eui_p50"] = example["climate_zone"].map(k12_benchmark)
    example["estimated_annual_kbtu"] = example["estimated_sqft"].astype(float) * example["site_eui_p50"]
    display(
        example[
            [
                "school_id",
                "school_name",
                "level",
                "climate_zone",
                "estimated_sqft",
                "site_eui_p50",
                "estimated_annual_kbtu",
            ]
        ]
    )

## 7. Save

The three datasets are written as CSV files and as three sheets of one Excel workbook. The school CSV is read back
to confirm that IDs kept their leading zeros.

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
datasets = {"schools": schools, "districts": districts, "bpd_eui_benchmarks": benchmarks}
csv_names = {
    "schools": "schools_2024_25.csv",
    "districts": "districts_2024_25.csv",
    "bpd_eui_benchmarks": "bpd_eui_benchmarks.csv",
}

for name, frame in datasets.items():
    frame.to_csv(PROCESSED_DIR / csv_names[name], index=False)

workbook_path = PROCESSED_DIR / "school_energy_datasets_2024_25.xlsx"
with pd.ExcelWriter(workbook_path) as workbook:
    for name, frame in datasets.items():
        frame.to_excel(workbook, sheet_name=name, index=False)

reread = pd.read_csv(PROCESSED_DIR / csv_names["schools"], **TEXT)
check(
    reread["school_id"].equals(schools["school_id"].reset_index(drop=True)),
    "Saved schools CSV changed school_id values.",
)
check(
    reread["district_id"].equals(schools["district_id"].reset_index(drop=True)),
    "Saved schools CSV changed district_id values.",
)

pd.DataFrame(
    {
        "rows": [len(frame) for frame in datasets.values()],
        "columns": [frame.shape[1] for frame in datasets.values()],
        "csv_megabytes": [
            round((PROCESSED_DIR / csv_names[name]).stat().st_size / 1e6, 1) for name in datasets
        ],
    },
    index=list(datasets),
)

## 8. Data dictionary

**`schools`** — every column from Step 1 (see the Step 1 data dictionary), plus:

| Column | From | Meaning |
|---|---|---|
| `footprint_file_available`, `building_found`, `footprint_sqft`, `building_capture_dates`, `nearest_building_distance_m`, `nearest_building_sqft`, `schools_at_same_point` | Step 2 | Building footprint match details (see Step 2) |
| `estimated_sqft` | Step 2 | Square footage to use; blank when not estimated |
| `sqft_method` | Step 2 | How `estimated_sqft` was set; also `not_operational` or `outside_square_footage_run` when blank for those reasons |
| `climate_zone` | Step 3 | ASHRAE climate zone; blank when not assigned |
| `climate_zone_method` | Step 3 | How the zone was assigned; `not_operational` for schools that are not operating |

**`districts`** — `district_id`, `district_name`, `admin_state`, `admin_state_name`, `schools` (all statuses),
`operational_schools`.

**`bpd_eui_benchmarks`** — as described in Step 4.